# FA Classification Dataset Creation

**Instructor-only notebook** -- generates `fa_classification_data.csv` for the Week 12 lab.

## Design Rationale

This notebook creates a simulated dataset of **60 participants**, each with summary FA rates across four conditions (attention, escape, tangible, play). Each participant has a known behavioral function, and their rates are generated so that the condition corresponding to their function is elevated.

### Function Distribution

- **Attention-maintained**: 20 participants
- **Escape-maintained**: 20 participants
- **Tangible-maintained**: 20 participants

No "automatic" function is included because it would require elevated play-condition rates, which creates interpretive complications for an introductory ML exercise.

### Rate Generation

For each participant:
- The **target condition** (matching their function) is drawn from a higher distribution (mean ~8--12, SD ~2).
- The **other conditions** are drawn from a lower distribution (mean ~1--3, SD ~1), clipped at 0.
- Noise is calibrated so that classification is achievable (~85--95% accuracy) but not trivial -- some participants have less clear differentiation, simulating real-world ambiguity.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(2024)

## Define Generation Parameters

In [ ]:
n_per_function = 20
functions = ["attention", "escape", "tangible"]
conditions = ["attention", "escape", "tangible", "play"]

# Rate parameters
high_mean = 10.0   # mean rate for the maintaining condition
high_sd = 2.5      # SD for the maintaining condition
low_mean = 2.0     # mean rate for non-maintaining conditions
low_sd = 1.2       # SD for non-maintaining conditions
play_mean = 0.8    # play/control is generally lowest
play_sd = 0.5

## Generate Participant Data

In [ ]:
rows = []
participant_id = 1

for function in functions:
    for _ in range(n_per_function):
        rates = {}
        for cond in conditions:
            if cond == function:
                # Elevated rate for maintaining condition
                rate = np.random.normal(high_mean, high_sd)
            elif cond == "play":
                # Play/control is always low
                rate = np.random.normal(play_mean, play_sd)
            else:
                # Other test conditions are low
                rate = np.random.normal(low_mean, low_sd)
            # Clip to non-negative and round
            rates[cond] = round(max(0.0, rate), 1)
        
        rows.append({
            "participant_id": participant_id,
            "attention_rate": rates["attention"],
            "escape_rate": rates["escape"],
            "tangible_rate": rates["tangible"],
            "play_rate": rates["play"],
            "function": function,
        })
        participant_id += 1

df = pd.DataFrame(rows)
print(f"Shape: {df.shape}")
print(f"\nFunction counts:\n{df['function'].value_counts()}")

## Verify Data Quality

Check that the maintaining condition tends to be elevated for each function group.

In [ ]:
print("Mean rates by function group:\n")
rate_cols = ["attention_rate", "escape_rate", "tangible_rate", "play_rate"]
summary = df.groupby("function")[rate_cols].mean().round(2)
print(summary)

print("\n--- Verification ---")
for func in functions:
    group = df[df["function"] == func]
    target_col = f"{func}_rate"
    other_cols = [c for c in rate_cols if c != target_col]
    target_mean = group[target_col].mean()
    other_mean = group[other_cols].values.mean()
    print(f"{func}: target condition mean = {target_mean:.2f}, other conditions mean = {other_mean:.2f}, ratio = {target_mean/other_mean:.1f}x")

## Preview and Save

In [ ]:
print(df.head(10).to_string(index=False))
print("...")
print(df.tail(5).to_string(index=False))

In [ ]:
df.to_csv("fa_classification_data.csv", index=False)
print("Saved fa_classification_data.csv")